In [2]:
%cd ..

F:\Projects\GraphRouterV2


C:\Users\CMG\AppData\Roaming\Python\Python312\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
import pandas as pd
import numpy as np

In [4]:
router_df = pd.read_csv('datasets/200Q-Final/router_training_data.csv')
router_df.head()

,row_id,query_id,task_description,task_description_embedding,query,query_embedding,Gold_Answer,Answer_Type,model,Predicted_Answer,...,Output_Tokens,Total_Tokens,Cost,Latency,Completion_Status,Error_Type,Final_Answer_Length,Timestamp,reasoning_score,response
0,80646b9ac545,44b2b96644c6,Divisibility + Pairwise Coprimality / Constrai...,"[[-0.004198932554572821, 0.067597396671772, -0...",The group of 10 girls should be divided into t...,"[[0.008862393908202648, -0.01544900517910719, ...",462,latex,openai/gpt-5,336,...,610,767,0.006296,10.213,success,wrong_answer,3,2026-06-22T11:33:24.397321+00:00,NaN,FINAL_ANSWER:\n336
1,d6e6c6d24304,44b2b96644c6,Divisibility + Pairwise Coprimality / Constrai...,"[[-0.004198932554572821, 0.067597396671772, -0...",The group of 10 girls should be divided into t...,"[[0.008862393908202648, -0.01544900517910719, ...",462,latex,anthropic/claude-sonnet-4-5,336,...,221,403,0.003861,6.291,success,wrong_answer,3,2026-06-22T11:33:44.699876+00:00,NaN,I need to find the number of ways to divide 10...
2,0b544c46ca65,44b2b96644c6,Divisibility + Pairwise Coprimality / Constrai...,"[[-0.004198932554572821, 0.067597396671772, -0...",The group of 10 girls should be divided into t...,"[[0.008862393908202648, -0.01544900517910719, ...",462,latex,qwen/qwen3-235b-a22b,$$,...,283,444,0.000588,5.144,success,wrong_answer,2,2026-06-22T11:34:47.171332+00:00,NaN,To divide 10 girls into two groups with **at l...
3,00c055cf04a2,44b2b96644c6,Divisibility + Pairwise Coprimality / Constrai...,"[[-0.004198932554572821, 0.067597396671772, -0...",The group of 10 girls should be divided into t...,"[[0.008862393908202648, -0.01544900517910719, ...",462,latex,meta-llama/llama-3.1-70b-instruct,\binom{10}{4} + \binom{10}{5},...,22,183,0.000073,1.730,success,none,29,2026-06-22T11:35:06.160700+00:00,NaN,FINAL_ANSWER: \binom{10}{4} + \binom{10}{5}
4,3a780bd77dc7,44b2b96644c6,Divisibility + Pairwise Coprimality / Constrai...,"[[-0.004198932554572821, 0.067597396671772, -0...",The group of 10 girls should be divided into t...,"[[0.008862393908202648, -0.01544900517910719, ...",462,latex,mistralai/mixtral-8x22b-instruct,115,...,11,186,0.000167,1.981,success,wrong_answer,3,2026-06-22T11:35:20.612191+00:00,NaN,FINAL_ANSWER:\n115


# Scoring Measure

In [5]:
import json
# File I/O functions
def loadjson(filename: str) -> dict:
    """
    Load data from a JSON file.

    Args:
        filename: Path to the JSON file

    Returns:
        Dictionary containing the loaded JSON data
    """
    with open(filename, 'r', encoding='utf-8') as file:
        data = json.load(file)
    return data

llm_path = "datasets/200Q-Final/LLM_Descriptions.json"
num_llms = 7
llm_description = loadjson(llm_path)
llm_names = list(llm_description.keys())

def _enforce_rectangular_query_blocks(df: pd.DataFrame) -> pd.DataFrame:
    """
    Guarantee the invariant every downstream positional index relies on:
    each query occupies exactly self.num_llms consecutive rows, one per
    distinct model value found in the router data. Groups by `query_id`
    (present in router_data.csv per Revised_Feature_Plan.md); falls back
    to grouping by raw `query` text with a warning if query_id is missing.

    The "complete set" of models is derived from the router dataset's own
    `model` column (self-consistency check), NOT compared against
    LLM_Descriptions.json's llm_names -- the two use unrelated naming
    schemes and matching them is a separate concern from data integrity.
    The one thing we do verify is that the router data's own model
    vocabulary has exactly self.num_llms distinct values, since that's
    the cardinality every downstream tensor shape assumes.

    Any query_id whose row set doesn't cover that vocabulary exactly once
    (missing run, duplicate run) is DROPPED WHOLESALE and reported --
    keeping it would silently misalign edge_org_id/edge_des_id/
    unique_index_list for every query after it, which is strictly worse
    than losing that one query's data.
    """
    group_col = "query_id" if "query_id" in df.columns else "query"
    if group_col == "query":
        print("[data_integrity][WARN] no query_id column found -- grouping by raw "
              "query text instead. If two different queries have identical text this "
              "will incorrectly merge them.")

    wanted_models = sorted(df["model"].unique().tolist())
    if len(wanted_models) != num_llms:
        raise ValueError(
            f"[data_integrity] router_data's `model` column has {len(wanted_models)} distinct "
            f"values {wanted_models}, but LLM_Descriptions.json declares {num_llms} LLMs "
            f"({llm_names}). These counts must match -- check for typos/extra models in "
            f"the data or a stale LLM_Descriptions.json."
        )
    wanted_models_set = set(wanted_models)
    llm_order = {name: i for i, name in enumerate(wanted_models)}  # stable order, data-derived

    kept_blocks = []
    n_dropped_queries = 0
    dropped_ids = []
    for key, group in df.groupby(group_col, sort=False):
        models_here = group["model"].tolist()
        if len(models_here) != num_llms or set(models_here) != wanted_models_set:
            n_dropped_queries += 1
            dropped_ids.append(key)
            continue
        kept_blocks.append(group.assign(_llm_sort=group["model"].map(llm_order)).sort_values("_llm_sort"))

    if n_dropped_queries:
        preview = dropped_ids[:10]
        print(f"[data_integrity][WARN] dropped {n_dropped_queries} incomplete/duplicate "
              f"query group(s) out of {df[group_col].nunique()} -- these queries did not "
              f"have exactly one row per model in {wanted_models}. Example query_id(s): {preview}"
              f"{' ...' if n_dropped_queries > 10 else ''}")

    if not kept_blocks:
        raise ValueError(
            "[data_integrity] 0 complete query blocks after filtering -- check the `model` "
            "column values in router_data.csv for typos/inconsistent naming: " + str(wanted_models)
        )

    result = pd.concat(kept_blocks, axis=0, ignore_index=True).drop(columns=["_llm_sort"])
    n_before, n_after = len(df), len(result)
    print(f"[data_integrity] rectangular-block check: {n_before} -> {n_after} rows "
          f"({n_before - n_after} dropped), {n_after // num_llms} complete queries retained.")
    return result

In [6]:
router_df = _enforce_rectangular_query_blocks(router_df)
router_df.shape

[data_integrity][WARN] dropped 22 incomplete/duplicate query group(s) out of 98 -- these queries did not have exactly one row per model in ['anthropic/claude-sonnet-4-5', 'deepseek/deepseek-r1', 'google/gemini-3.5-flash', 'meta-llama/llama-3.1-70b-instruct', 'mistralai/mixtral-8x22b-instruct', 'openai/gpt-5', 'qwen/qwen3-235b-a22b']. Example query_id(s): ['8ff9e9f62f23', 'e95fa790cb93', '9097abd21c8e', '7208ad940d2c', '437120961e45', '8600dcc5546d', '70629b6e7c5d', '84c96fbdf01d', '015197614be4', '290f22cd04ea'] ...
[data_integrity] rectangular-block check: 664 -> 532 rows (132 dropped), 76 complete queries retained.


(532, 22)

In [7]:
router_df

,row_id,query_id,task_description,task_description_embedding,query,query_embedding,Gold_Answer,Answer_Type,model,Predicted_Answer,...,Output_Tokens,Total_Tokens,Cost,Latency,Completion_Status,Error_Type,Final_Answer_Length,Timestamp,reasoning_score,response
0,d6e6c6d24304,44b2b96644c6,Divisibility + Pairwise Coprimality / Constrai...,"[[-0.004198932554572821, 0.067597396671772, -0...",The group of 10 girls should be divided into t...,"[[0.008862393908202648, -0.01544900517910719, ...",462,latex,anthropic/claude-sonnet-4-5,336,...,221,403,0.003861,6.291,success,wrong_answer,3,2026-06-22T11:33:44.699876+00:00,NaN,I need to find the number of ways to divide 10...
1,5832db75e144,44b2b96644c6,Divisibility + Pairwise Coprimality / Constrai...,"[[-0.004198932554572821, 0.067597396671772, -0...",The group of 10 girls should be divided into t...,"[[0.008862393908202648, -0.01544900517910719, ...",462,latex,deepseek/deepseek-r1,336,...,5841,5998,0.014712,204.084,success,wrong_answer,3,2026-06-22T12:52:07.306918+00:00,NaN,The number of ways to divide 10 distinct girls...
2,acca4d08183e,44b2b96644c6,Divisibility + Pairwise Coprimality / Constrai...,"[[-0.004198932554572821, 0.067597396671772, -0...",The group of 10 girls should be divided into t...,"[[0.008862393908202648, -0.01544900517910719, ...",462,latex,google/gemini-3.5-flash,336,...,786,948,0.007317,5.973,success,wrong_answer,3,2026-06-22T12:51:49.776277+00:00,NaN,FINAL_ANSWER:\n336
3,00c055cf04a2,44b2b96644c6,Divisibility + Pairwise Coprimality / Constrai...,"[[-0.004198932554572821, 0.067597396671772, -0...",The group of 10 girls should be divided into t...,"[[0.008862393908202648, -0.01544900517910719, ...",462,latex,meta-llama/llama-3.1-70b-instruct,\binom{10}{4} + \binom{10}{5},...,22,183,0.000073,1.730,success,none,29,2026-06-22T11:35:06.160700+00:00,NaN,FINAL_ANSWER: \binom{10}{4} + \binom{10}{5}
4,3a780bd77dc7,44b2b96644c6,Divisibility + Pairwise Coprimality / Constrai...,"[[-0.004198932554572821, 0.067597396671772, -0...",The group of 10 girls should be divided into t...,"[[0.008862393908202648, -0.01544900517910719, ...",462,latex,mistralai/mixtral-8x22b-instruct,115,...,11,186,0.000167,1.981,success,wrong_answer,3,2026-06-22T11:35:20.612191+00:00,NaN,FINAL_ANSWER:\n115
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
527,84a94d01efc4,2347acb5eb3b,Olympiad Reasoning,"[[-0.00014781509526073933, 0.14556902647018433...",Find the largest possible real part of \[(75+1...,"[[0.018808305263519287, 0.11890657246112823, -...",540,latex,google/gemini-3.5-flash,540,...,747,892,0.006940,4.365,success,none,3,2026-07-01T00:47:14.814064+00:00,NaN,FINAL_ANSWER:\n540
528,c0fbd2d28827,2347acb5eb3b,Olympiad Reasoning,"[[-0.00014781509526073933, 0.14556902647018433...",Find the largest possible real part of \[(75+1...,"[[0.018808305263519287, 0.11890657246112823, -...",540,latex,meta-llama/llama-3.1-70b-instruct,\sqrt{117^2+144^2}+75\cdot4,...,21,163,0.000065,3.502,success,none,27,2026-07-01T00:47:49.400257+00:00,NaN,FINAL_ANSWER:\n\sqrt{117^2+144^2}+75\cdot4
529,a938cced6691,2347acb5eb3b,Olympiad Reasoning,"[[-0.00014781509526073933, 0.14556902647018433...",Find the largest possible real part of \[(75+1...,"[[0.018808305263519287, 0.11890657246112823, -...",540,latex,mistralai/mixtral-8x22b-instruct,135,...,11,167,0.000150,1.170,success,wrong_answer,3,2026-07-01T00:47:54.936346+00:00,NaN,FINAL_ANSWER:\n135
530,adb08f542d48,2347acb5eb3b,Olympiad Reasoning,"[[-0.00014781509526073933, 0.14556902647018433...",Find the largest possible real part of \[(75+1...,"[[0.018808305263519287, 0.11890657246112823, -...",540,latex,openai/gpt-5,540,...,1420,1560,0.014375,28.248,success,none,3,2026-06-30T23:27:38.981802+00:00,NaN,FINAL_ANSWER:\n540


In [8]:
from data_processing import feature_builder as fb

encoders = fb.fit_encoders(
    router_df,
)

[feature_builder] 'Answer_Type' is constant ('latex') in the train split -- dropped (0 dims) instead of encoded as a dead-weight one-hot column.


In [9]:
encoders

FeatureEncoders(domain_categories=['Combinatorics_Discrete', 'Number_Theory', 'Algebra', 'Geometry', 'Olympiad_Logic', 'Other'], domain_precomputed_columns=[], reasoning_type_categories=[], difficulty_categories=[], difficulty_ordinal_order=None, difficulty_is_numeric=False, difficulty_mu_sigma=(0.0, 1.0), reasoning_depth_categories=[], reasoning_depth_ordinal_order=None, reasoning_depth_is_numeric=False, reasoning_depth_mu_sigma=(0.0, 1.0), answer_type_categories=[], solution_steps_mu_sigma=(0.0, 1.0), question_length_mu_sigma=(5.580089635700915, 0.6565240740686119), completion_status_categories=['success'], error_type_categories=['no_final_answer', 'none', 'wrong_answer'], cost_mu_sigma=(0.015854302011278198, 0.035626543814508485), latency_mu_sigma=(79.98872744360904, 222.38950959158626), input_tokens_mu_sigma=(274.49436090225566, 1107.6739709445421), output_tokens_mu_sigma=(2716.281954887218, 5789.296793576096), completion_reliability_by_model={'anthropic/claude-sonnet-4-5': 0.0, 'd

In [10]:
utility_weights = {
  "w_success": 1.0,
  "w_cost": 0.3,
  "w_latency": 0.3,
  "w_output_tokens": 0.2,
  "w_completion_reliability": 0.5
    }

In [11]:
len(router_df.model.unique())

7

In [12]:
edge_features = fb.build_edge_features(router_df, encoders)
edge_features

array([[ 0.        , -0.3366395 , -0.3313903 , ...,  0.        ,
         0.        ,  1.        ],
       [ 0.        , -0.032052  ,  0.5580087 , ...,  0.        ,
         0.        ,  1.        ],
       [ 0.        , -0.23963319, -0.33282024, ...,  0.        ,
         0.        ,  1.        ],
       ...,
       [ 0.        , -0.440795  , -0.35441747, ...,  0.        ,
         0.        ,  1.        ],
       [ 1.        , -0.04152247, -0.23265813, ...,  0.        ,
         1.        ,  0.        ],
       [ 1.        , -0.3036346 , -0.2240606 , ...,  0.        ,
         1.        ,  0.        ]], dtype=float32)

In [13]:
utility_list = fb.compute_utility(router_df, encoders, **utility_weights)
utility_list

array([ 2.86612213e-01, -2.65735090e-01,  2.38420531e-01,  1.33153558e+00,
        3.30783755e-01,  2.47376442e-01,  3.13575834e-01,  2.16807097e-01,
       -9.80435610e-01,  1.71236560e-01,  1.33137298e+00,  1.33004844e+00,
        1.20127141e+00,  1.30528224e+00,  2.15351030e-01, -6.03930891e-01,
        1.27695918e-01,  3.32691073e-01,  3.30899507e-01,  2.68264115e-01,
        3.30785602e-01,  1.21945786e+00,  2.78333724e-01,  1.10919797e+00,
        3.32728237e-01,  3.30041587e-01,  3.24955732e-02,  2.90302843e-01,
        1.26425862e+00,  1.21454120e+00,  1.27155805e+00,  1.32555819e+00,
        1.32704103e+00,  1.25782645e+00,  3.13551426e-01,  1.25015557e+00,
        7.08641231e-01,  1.25199986e+00,  3.32814276e-01,  3.28209281e-01,
        1.22744668e+00,  1.29980445e+00,  1.47536650e-01,  7.75919437e-01,
        1.08020163e+00,  1.32810223e+00,  3.31502885e-01,  1.09791017e+00,
        1.22963345e+00,  1.75550237e-01,  2.26218104e-01,  1.79758862e-01,
        1.33048785e+00,  

In [14]:
num_llms = 7
SOFTMAX_TEMPERATURE = 1.0
utility_reshaped = utility_list.reshape(-1, num_llms) / SOFTMAX_TEMPERATURE
utility_reshaped

array([[ 2.86612213e-01, -2.65735090e-01,  2.38420531e-01,
         1.33153558e+00,  3.30783755e-01,  2.47376442e-01,
         3.13575834e-01],
       [ 2.16807097e-01, -9.80435610e-01,  1.71236560e-01,
         1.33137298e+00,  1.33004844e+00,  1.20127141e+00,
         1.30528224e+00],
       [ 2.15351030e-01, -6.03930891e-01,  1.27695918e-01,
         3.32691073e-01,  3.30899507e-01,  2.68264115e-01,
         3.30785602e-01],
       [ 1.21945786e+00,  2.78333724e-01,  1.10919797e+00,
         3.32728237e-01,  3.30041587e-01,  3.24955732e-02,
         2.90302843e-01],
       [ 1.26425862e+00,  1.21454120e+00,  1.27155805e+00,
         1.32555819e+00,  1.32704103e+00,  1.25782645e+00,
         3.13551426e-01],
       [ 1.25015557e+00,  7.08641231e-01,  1.25199986e+00,
         3.32814276e-01,  3.28209281e-01,  1.22744668e+00,
         1.29980445e+00],
       [ 1.47536650e-01,  7.75919437e-01,  1.08020163e+00,
         1.32810223e+00,  3.31502885e-01,  1.09791017e+00,
         1.2296334

In [15]:
utility_reshaped[0]

array([ 0.2866122 , -0.2657351 ,  0.23842053,  1.3315356 ,  0.33078375,
        0.24737644,  0.31357583], dtype=float32)

In [15]:
edge_features[0]

array([ 0.        , -0.27179012, -0.30002776, -0.10013796, -0.34709436,
        1.        ,  0.        ,  0.        ,  1.        ], dtype=float32)

In [26]:
import pandas as pd
import numpy as np

def calculate_query_utility(
    query_id: str | int,
    df: pd.DataFrame,
    weights: dict = None
) -> pd.DataFrame:
    """
    Calculates utility scores across all models for a specific query ID using
    exact dataset column names.

    Parameters:
    -----------
    query_id : str or int
        Unique identifier for the target query.
    df : pd.DataFrame
        DataFrame containing query benchmarking records across candidate models.
    weights : dict, optional
        Utility weights matching config.yaml specifications.

    Returns:
    --------
    pd.DataFrame
        Table of candidate models ranked by calculated utility score.
    """
    # Default weights matching config.yaml
    if weights is None:
        weights = {
            'w_success': 1.0,
            'w_cost': 0.3,
            'w_latency': 0.3,
            'w_output_tokens': 0.2,
            'w_completion_reliability': 0.5
        }

    # Filter dataframe for the given query
    query_df = df[df['query_id'] == query_id].copy()
    if query_df.empty:
        raise ValueError(f"Query ID '{query_id}' not found in the dataset.")

    # Extract key metric series using exact column names
    success = query_df['Correct'].astype(float)
    cost = query_df['Cost'].astype(float)
    latency = query_df['Latency'].astype(float)
    out_tokens = query_df['Output_Tokens'].astype(float)

    # Reliability: 1.0 if status indicates success/completion, otherwise 0.0
    reliability = query_df['Completion_Status'].apply(
        lambda x: 1.0 if x in [1, True, 'Success', 'SUCCESS', 'Completed', 'completed'] else (float(x) if str(x).replace('.','',1).isdigit() else 0.0)
    )

    # Per-query min-max normalization function for penalty metrics
    def min_max_norm(series: pd.Series) -> pd.Series:
        rng = series.max() - series.min()
        return (series - series.min()) / rng if rng > 0 else pd.Series(0.0, index=series.index)

    # Calculate normalized penalties relative to candidate models for this query
    cost_norm = min_max_norm(cost)
    latency_norm = min_max_norm(latency)
    out_tokens_norm = min_max_norm(out_tokens)

    # Utility score calculation
    query_df['utility_score'] = (
        weights['w_success'] * success
        + weights['w_completion_reliability'] * reliability
        - weights['w_cost'] * cost_norm
        - weights['w_latency'] * latency_norm
        - weights['w_output_tokens'] * out_tokens_norm
    )

    # Select key summary columns and sort by highest utility score
    display_cols = [
        'model', 'utility_score', 'Correct', 'Cost',
        'Latency', 'Output_Tokens', 'Completion_Status'
    ]

    result = (
        query_df[display_cols]
        .sort_values(by='utility_score', ascending=False)
        .reset_index(drop=True)
    )

    return result

In [27]:
router_df

,row_id,query_id,task_description,task_description_embedding,query,query_embedding,Gold_Answer,Answer_Type,model,Predicted_Answer,...,Output_Tokens,Total_Tokens,Cost,Latency,Completion_Status,Error_Type,Final_Answer_Length,Timestamp,reasoning_score,response
0,d6e6c6d24304,44b2b96644c6,Divisibility + Pairwise Coprimality / Constrai...,"[[-0.004198932554572821, 0.067597396671772, -0...",The group of 10 girls should be divided into t...,"[[0.008862393908202648, -0.01544900517910719, ...",462,latex,anthropic/claude-sonnet-4-5,336,...,221,403,0.003861,6.291,success,wrong_answer,3,2026-06-22T11:33:44.699876+00:00,NaN,I need to find the number of ways to divide 10...
1,5832db75e144,44b2b96644c6,Divisibility + Pairwise Coprimality / Constrai...,"[[-0.004198932554572821, 0.067597396671772, -0...",The group of 10 girls should be divided into t...,"[[0.008862393908202648, -0.01544900517910719, ...",462,latex,deepseek/deepseek-r1,336,...,5841,5998,0.014712,204.084,success,wrong_answer,3,2026-06-22T12:52:07.306918+00:00,NaN,The number of ways to divide 10 distinct girls...
2,acca4d08183e,44b2b96644c6,Divisibility + Pairwise Coprimality / Constrai...,"[[-0.004198932554572821, 0.067597396671772, -0...",The group of 10 girls should be divided into t...,"[[0.008862393908202648, -0.01544900517910719, ...",462,latex,google/gemini-3.5-flash,336,...,786,948,0.007317,5.973,success,wrong_answer,3,2026-06-22T12:51:49.776277+00:00,NaN,FINAL_ANSWER:\n336
3,00c055cf04a2,44b2b96644c6,Divisibility + Pairwise Coprimality / Constrai...,"[[-0.004198932554572821, 0.067597396671772, -0...",The group of 10 girls should be divided into t...,"[[0.008862393908202648, -0.01544900517910719, ...",462,latex,meta-llama/llama-3.1-70b-instruct,\binom{10}{4} + \binom{10}{5},...,22,183,0.000073,1.730,success,none,29,2026-06-22T11:35:06.160700+00:00,NaN,FINAL_ANSWER: \binom{10}{4} + \binom{10}{5}
4,3a780bd77dc7,44b2b96644c6,Divisibility + Pairwise Coprimality / Constrai...,"[[-0.004198932554572821, 0.067597396671772, -0...",The group of 10 girls should be divided into t...,"[[0.008862393908202648, -0.01544900517910719, ...",462,latex,mistralai/mixtral-8x22b-instruct,115,...,11,186,0.000167,1.981,success,wrong_answer,3,2026-06-22T11:35:20.612191+00:00,NaN,FINAL_ANSWER:\n115
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
527,84a94d01efc4,2347acb5eb3b,Olympiad Reasoning,"[[-0.00014781509526073933, 0.14556902647018433...",Find the largest possible real part of \[(75+1...,"[[0.018808305263519287, 0.11890657246112823, -...",540,latex,google/gemini-3.5-flash,540,...,747,892,0.006940,4.365,success,none,3,2026-07-01T00:47:14.814064+00:00,NaN,FINAL_ANSWER:\n540
528,c0fbd2d28827,2347acb5eb3b,Olympiad Reasoning,"[[-0.00014781509526073933, 0.14556902647018433...",Find the largest possible real part of \[(75+1...,"[[0.018808305263519287, 0.11890657246112823, -...",540,latex,meta-llama/llama-3.1-70b-instruct,\sqrt{117^2+144^2}+75\cdot4,...,21,163,0.000065,3.502,success,none,27,2026-07-01T00:47:49.400257+00:00,NaN,FINAL_ANSWER:\n\sqrt{117^2+144^2}+75\cdot4
529,a938cced6691,2347acb5eb3b,Olympiad Reasoning,"[[-0.00014781509526073933, 0.14556902647018433...",Find the largest possible real part of \[(75+1...,"[[0.018808305263519287, 0.11890657246112823, -...",540,latex,mistralai/mixtral-8x22b-instruct,135,...,11,167,0.000150,1.170,success,wrong_answer,3,2026-07-01T00:47:54.936346+00:00,NaN,FINAL_ANSWER:\n135
530,adb08f542d48,2347acb5eb3b,Olympiad Reasoning,"[[-0.00014781509526073933, 0.14556902647018433...",Find the largest possible real part of \[(75+1...,"[[0.018808305263519287, 0.11890657246112823, -...",540,latex,openai/gpt-5,540,...,1420,1560,0.014375,28.248,success,none,3,2026-06-30T23:27:38.981802+00:00,NaN,FINAL_ANSWER:\n540


In [32]:
router_df.query_id.unique()

array(['44b2b96644c6', '2ccbe9e61ecf', '6d70005c9303', 'df7389706c09',
       'b1ea61565b4e', '619336412192', '321d86ce79e1', 'b3861445cc06',
       'd399e6ff9ba0', 'fa1f02a208b2', '98a313a86f15', 'f5c44c1f20cd',
       'da25610ff924', 'ba9152f3d981', 'a17f289c902d', 'e21531fb7bff',
       'dd187547d811', '93ee90bd4810', 'ba20256f2d8f', '552752099497',
       '2822569975bb', 'f3d797442d90', '5e99c1f8e33f', '5cc894a0e8a6',
       'ac19f76765b0', '4d866680f923', 'b80bbab3f94d', 'e6ea64827cab',
       'f2f2d976fb29', '77bffe9f7e54', 'bce9ef2f43a9', 'f3d7b78d243a',
       'f969f616feb3', '9cdfb2e553b3', '6fda1b293f84', 'c4ae027f8486',
       '6a5b9b9200b7', 'fd14e1d88f3a', 'a69cadd900e6', '9e41363fe398',
       '1ee995a8d3c7', '11b94562acc8', 'c44d2bf7affb', '7108276b8fa4',
       'cdb9abd4f7a1', '7968ea2f49d4', '9a742f03e053', 'b55b05e56d89',
       '2813fc865fdb', 'ad13376b6795', 'c2359e2c8745', '80a7415f3d3c',
       'c3c962ab1ff1', '0e2fbc9ece85', '95aedf7c6ac6', '75e19ca5a482',
      

In [43]:
# Custom weights if needed (or pass None to use defaults)
custom_weights = {
    'w_success': 1.0,
    'w_cost': 0.6,
    'w_latency': 0.3,
    'w_output_tokens': 0.2,
    'w_completion_reliability': 0.0
}

# Calculate utility scores across models for query ID 'Q_104'
utility_table = calculate_query_utility(query_id="77bffe9f7e54", df=router_df, weights=custom_weights)
print(utility_table)

                               model  utility_score  Correct      Cost  \
0        anthropic/claude-sonnet-4-5       0.546856        1  0.009678   
1                       openai/gpt-5       0.458181        1  0.010765   
2            google/gemini-3.5-flash       0.318548        1  0.014018   
3  meta-llama/llama-3.1-70b-instruct      -0.000128        0  0.000073   
4   mistralai/mixtral-8x22b-instruct      -0.004737        0  0.000180   
5               deepseek/deepseek-r1      -0.019699        1  0.012151   
6               qwen/qwen3-235b-a22b      -0.054705        0  0.000793   

   Latency  Output_Tokens Completion_Status  
0    9.509            619           success  
1   22.681           1062           success  
2   11.380           1538           success  
3    1.437             10           success  
4    1.365             13           success  
5  169.790           4814           success  
6    5.761            392           success  
